# 10 - ETa dynamics by crop type

Uses the daily ETa rasters from notebook 09 and the field boundaries of the 18 long-term plots
(with the crop grown in each year) to build daily ETa time series for wheat, corn, millet and fallow,
2019-2025. Produces Figs. 10 and 11 of the paper.

In [ ]:
import re
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rasterio.mask import mask
from shapely.geometry import box, mapping

In [ ]:
FIELDS_SHP = Path("../data/raw/shapefiles/crop_fields/Crop_types_2019_2025.shp")
RASTER_DIR = Path("../results/eta_rasters")
CLIMATE_CSV = Path("../data/raw/climate/CoAgMet_Climatic_Data.csv")
SUMMARY_DIR = Path("../results/eta_summary")
FIG_DIR = Path("../results/figures")
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# crop codes used in the Year_YYYY columns of the field shapefile
CROPS = {"W": "Wheat", "C": "Corn", "M": "Millet", "F": "Fallow", "FM": "FallowMillet"}
YEARS = range(2019, 2026)

## Pixel statistics per crop and day

For each daily raster, the fields under each crop in that year are selected and the mean, median,
max and min ETa of their pixels are recorded. One Excel file per year, one sheet per crop.

In [ ]:
fields = gpd.read_file(FIELDS_SHP)
year_cols = {int(c.split("_")[1]): c for c in fields.columns if c.startswith("Year_")}

rasters = []
for fp in RASTER_DIR.glob("*.tif"):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", fp.name)
    if m:
        d = datetime.strptime(m.group(1), "%Y-%m-%d")
        if d.year in YEARS:
            rasters.append((d, fp))
if not rasters:
    raise RuntimeError(f"no ETa rasters for {YEARS.start}-{YEARS.stop - 1} in {RASTER_DIR}")
rasters.sort()

with rasterio.open(rasters[0][1]) as src:
    fields = fields.to_crs(src.crs)

stats = {yr: {s: {} for s in ("mean", "median", "max", "min")} for yr in year_cols}

for day, tif in rasters:
    col = year_cols.get(day.year)
    if col is None:
        print(f"no Year_{day.year} column, skipping {day.date()}")
        continue

    with rasterio.open(tif) as src:
        bounds = box(*src.bounds)
        for code_, crop in CROPS.items():
            sel = fields[(fields[col] == code_) & fields.geometry.intersects(bounds)]
            if sel.empty:
                vals = np.array([])
            else:
                out, _ = mask(src, [mapping(g) for g in sel.geometry],
                              crop=False, all_touched=True, nodata=np.nan)
                band = out[0].astype(float)
                vals = band[np.isfinite(band)]

            if vals.size:
                values = (vals.mean(), np.median(vals), vals.max(), vals.min())
            else:
                values = (np.nan,) * 4
            for s, v in zip(("mean", "median", "max", "min"), values):
                stats[day.year][s].setdefault(day, {})[crop] = float(v)

for yr, ydict in stats.items():
    dates = sorted(ydict["mean"])
    if not dates:
        continue
    with pd.ExcelWriter(SUMMARY_DIR / f"EtA_summary_{yr}.xlsx", engine="openpyxl") as w:
        for crop in ydict["mean"][dates[0]]:
            out = pd.DataFrame({s: [ydict[s][d].get(crop, np.nan) for d in dates]
                                for s in ("mean", "median", "max", "min")}, index=dates)
            out.index.name = "Date"
            out.to_excel(w, sheet_name=crop[:31])
    print(f"wrote EtA_summary_{yr}.xlsx")

## Daily ETa and 10-day antecedent precipitation (Fig. 10)

Precipitation bars hang from the top axis. Dashed lines mark typical planting (green) and
harvest (black) dates for each crop.

In [ ]:
precip = pd.read_csv(CLIMATE_CSV, parse_dates=["Date"]).set_index("Date")
years = list(YEARS)
crops = ["Wheat", "Corn", "Millet", "Fallow"]


def crop_dates(year, crop):
    if crop == "Wheat":    # winter wheat, planted the autumn before
        return [(pd.Timestamp(year - 1, 9, 1), "plant"), (pd.Timestamp(year, 7, 1), "harvest")]
    if crop == "Corn":
        return [(pd.Timestamp(year, 5, 1), "plant"), (pd.Timestamp(year, 10, 15), "harvest")]
    if crop == "Millet":
        return [(pd.Timestamp(year, 6, 1), "plant"), (pd.Timestamp(year, 9, 15), "harvest")]
    return []


books = {yr: pd.read_excel(SUMMARY_DIR / f"EtA_summary_{yr}.xlsx", sheet_name=None, parse_dates=["Date"])
         for yr in years}
max_eta = max(df["mean"].max() for b in books.values() for df in b.values())
max_pr = precip["10-d pcp"].max()

PLANT = dict(color="green", linestyle="--", linewidth=2.0, alpha=0.95, zorder=0)
HARVEST = dict(color="black", linestyle="--", linewidth=2.0, alpha=0.95, zorder=0)

fig, axes = plt.subplots(len(years), len(crops), figsize=(4 * len(crops), 3 * len(years)))
fig.patch.set_facecolor("white")

for i, yr in enumerate(years):
    pr = precip.loc[precip.index.year == yr, "10-d pcp"]
    start, end = pd.Timestamp(f"{yr}-01-01"), pd.Timestamp(f"{yr}-12-31")

    for j, crop in enumerate(crops):
        ax = axes[i, j]

        ax2 = ax.twinx()
        ax2.set_xlim(start, end)
        ax2.set_ylim(max_pr * 2.0, 0)        # inverted, bars hang from the top
        ax2.bar(pr.index, pr, width=0.5, color="lightcoral", alpha=0.7, zorder=1)
        if j == len(crops) - 1:
            ax2.set_ylabel("10-d Pcp (mm)", rotation=270, labelpad=15, fontsize=14)
            ax2.tick_params(labelsize=12)
        else:
            ax2.tick_params(right=False, labelright=False, length=0)

        s = books[yr][crop].sort_values("Date").set_index("Date")
        ax.set_xlim(start, end)
        ax.set_ylim(0, max_eta * 1.2)
        ax.set_zorder(3)
        ax.patch.set_alpha(0)
        eta_line = ax.plot(s.index, s["mean"], linewidth=1.5, color="steelblue", label="ETa")[0]

        for d, kind in crop_dates(yr, crop):
            ax.axvline(d, **(PLANT if kind == "plant" else HARVEST))

        if i == 0:
            ax.set_title(crop, pad=6, fontweight="bold", fontsize=16)
        if i == len(years) - 1:
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
            ax.tick_params(axis="x", labelsize=12)
        else:
            ax.tick_params(labelbottom=False)
        if j == 0:
            ax.set_ylabel("ETa (mm/day)", fontsize=14)
            ax.tick_params(axis="y", labelsize=12)
            ax.text(-0.2, 0.5, str(yr), transform=ax.transAxes, rotation="vertical",
                    fontweight="bold", fontsize=16, va="center", ha="center")
        else:
            ax.tick_params(labelleft=False)

        if i == 0 and j == len(crops) - 1:
            ax.legend(handles=[
                eta_line,
                Patch(facecolor="lightcoral", edgecolor="lightcoral", label="10-day Pcp"),
                Line2D([0], [0], **{**PLANT, "alpha": 1.0}, label="Planting"),
                Line2D([0], [0], **{**HARVEST, "alpha": 1.0}, label="Harvesting"),
            ], loc="upper right", bbox_to_anchor=(0.95, 0.90), fontsize=14, frameon=False)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig10_daily_eta_by_crop.png", dpi=300, bbox_inches="tight")
plt.show()

## Monthly precipitation and ETa by crop (Fig. 11)

Monthly ETa is the sum of the daily field-mean ETa, averaged over 2019-2025. Monthly precipitation
is averaged over all years in the weather file.

In [ ]:
clim = pd.read_csv(CLIMATE_CSV, parse_dates=["Date"])
clim["Month"] = clim["Date"].dt.month
monthly_p = clim.groupby([clim["Date"].dt.year, "Month"])["Precipitation"].sum().reset_index()
table = monthly_p.groupby("Month")["Precipitation"].mean().rename("P (mm)").to_frame()

labels = {"Wheat": "ET-W", "Corn": "ET-C", "Millet": "ET-M", "Fallow": "ET-F"}
for crop, label in labels.items():
    per_year = []
    for f in sorted(SUMMARY_DIR.glob("*.xlsx")):
        s = pd.read_excel(f, sheet_name=crop, parse_dates=["Date"])
        s["Month"] = s["Date"].dt.month
        per_year.append(s.groupby([s["Date"].dt.year, "Month"])["mean"].sum().reset_index())
    table[label] = pd.concat(per_year).groupby("Month")["mean"].mean()

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
x = np.arange(12)
width = 0.15
colors = ["#4B9CD3", "#F28E2B", "#59A14F", "#E15759", "#B07AA1"]

plt.figure(figsize=(12, 5))
p_bars = plt.bar(x - 2 * width, table["P (mm)"], width=width, label="P (mm)", color=colors[0],
                 hatch="//", edgecolor="black", linewidth=0.6)
for k, label in enumerate(labels.values()):
    plt.bar(x + (k - 1) * width, table[label], width=width, label=label,
            color=colors[k + 1], edgecolor="black", linewidth=0.5)

for rect in p_bars:
    h = rect.get_height()
    if h > 0:
        plt.text(rect.get_x() + rect.get_width() / 2, h + 2, f"{h:.0f}",
                 ha="center", va="bottom", fontsize=10, rotation=90)

plt.xticks(x, months, fontsize=11)
plt.ylabel("P, ETa (mm)", fontsize=12)
plt.ylim(0, table.max().max() * 1.2)
plt.legend(frameon=True, fontsize=10, loc="upper left", ncol=3, edgecolor="black")
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig11_monthly_p_eta.png", dpi=300, bbox_inches="tight")
plt.show()

table.round(1)